# Maia Lite 0.3.0 — SSD-first Corpus v8 & Checkpoint Staging v1

**Política obrigatória:** Drive somente para sincronização de entrada/saída; SQLite, shards, tokenizer, logs e checkpoints ativos **sempre no SSD `/content/maia`**. Este notebook é um **preflight CPU**, não treina com o corpus. Não selecionar GPU para executar esta etapa. Em toda sessão nova, execute o staging novamente antes de carregar o modelo.

**Segurança de direitos:** inspeção e cópia não significam aprovação para treinamento. O leitor real é restrito a inspeção e o acesso para treinamento depende de um manifesto de autorização separado e explícito. Nenhum gate é automaticamente aprovado.

**Nota operacional:** o corpus tem ~29,55 GB binários e **60.612 arquivos**. A primeira cópia pode demorar; o notebook registra progresso, permite retomada da cópia na mesma sessão e falha se faltar espaço. A transferência inicial também consome tempo de sessão. Em runtime novo, `/content` é apagado e a cópia deve ser refeita.

In [ ]:
import os, sys, json, time, math, shutil, sqlite3, hashlib, tempfile, subprocess
from pathlib import Path
from collections import Counter
import numpy as np

DRIVE_PROJECT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
DRIVE_PRODUCTION = DRIVE_PROJECT/'corpus_v8/runs/acquisition_v8_5_001/pretraining_production_v1_full'
DRIVE_DB = DRIVE_PRODUCTION/'recovery_audit_v4/state_operational_v5.sqlite3'
LOCAL_ROOT = Path('/content/maia')
LOCAL_CORPUS = LOCAL_ROOT/'corpus_v8'
LOCAL_DB = LOCAL_CORPUS/'state_operational_v5.sqlite3'
LOCAL_SHARDS = LOCAL_CORPUS/'shards'
LOCAL_CKPTS = LOCAL_ROOT/'checkpoints'
DRIVE_CKPTS = DRIVE_PROJECT/'pretraining_05b_ssd_first_checkpoints'
BLOCK_SIZE=2048; VOCAB_SIZE=50257; BYTES_PER_BLOCK=8192
EXPECTED_CATALOG_HASH='737afb495abd1f1b1947c85fc85255615e53473002032bfb904537a4cee54985'

# Ajustar somente se a pasta do projeto mudar. Não ligar a GPU para este notebook.
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
except ImportError:
    print('Fora do Colab: monte o Drive ou ajuste DRIVE_PROJECT manualmente.')
assert DRIVE_DB.is_file(), f'Banco ausente: {DRIVE_DB}'
LOCAL_CORPUS.mkdir(parents=True,exist_ok=True)
LOCAL_CKPTS.mkdir(parents=True,exist_ok=True)
print('SSD:',LOCAL_ROOT,'Drive source:',DRIVE_PRODUCTION,flush=True)

## 1. Inventário rápido do Drive — sem `PRAGMA quick_check`

Apenas uma consulta SQL; não abre nenhum shard e não exige GPU. O hash é calculado sobre os caminhos **originais** para manter a identidade estável mesmo após o remapeamento.

In [ ]:
t0=time.monotonic()
con=sqlite3.connect(DRIVE_DB.resolve().as_uri()+'?mode=ro',uri=True)
try:
    records=con.execute('SELECT source,split,shard_index,path,blocks,sha256 FROM shard ORDER BY source,split,shard_index').fetchall()
finally: con.close()
assert len(records)==60612, f'Quantidade inesperada: {len(records)}'
assert all(isinstance(r[5],str) and len(r[5])==64 and int(r[4])>0 for r in records)
CATALOG_HASH=hashlib.sha256(json.dumps(records,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
assert CATALOG_HASH==EXPECTED_CATALOG_HASH, f'CATALOG CHANGED: {CATALOG_HASH}'
assert sum(int(r[4]) for r in records if r[1]=='train')==3390434

def local_path_from_record(r):
    original=Path(r[3])
    # Proteção contra path traversal e paths inesperados.
    rel=original.relative_to(DRIVE_PRODUCTION)
    assert rel.parts and rel.parts[0]=='shards' and '..' not in rel.parts, f'Caminho inválido: {original}'
    dest=(LOCAL_CORPUS/rel).resolve()
    assert dest.is_relative_to(LOCAL_SHARDS.resolve()), f'Destino inválido: {dest}'
    return dest

for r in records: local_path_from_record(r)
TOTAL_BYTES=sum(int(r[4])*BYTES_PER_BLOCK for r in records)
print('Catalog hash:',CATALOG_HASH)
print('Shards:',len(records),'Blocks:',sum(int(r[4]) for r in records))
print('Dataset GiB:',round(TOTAL_BYTES/2**30,2),'Catalog seconds:',round(time.monotonic()-t0,2))
print('CATALOG PREFLIGHT: PASS',flush=True)

## 2. Espaço livre e cópia segura para SSD

Copia em `.part` e renomeia somente após validar tamanho **e SHA-256**; mantém um arquivo de progresso local. Um arquivo local com tamanho correto é verificado por hash antes de ser reutilizado. A cópia não altera o Drive. Para reduzir custo, verificamos o hash do destino uma vez por execução; isso lê o SSD, não o Drive.

**Importante:** 60 mil arquivos podem causar latência na cópia inicial. O progresso aparece a cada 250 arquivos e 30 segundos. Se houver um arquivo faltando ou checksum incorreto, a execução falha. O espaço adicional inclui margem para checkpoints e temporários.

In [ ]:
def sha256_file(path,chunk_size=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(chunk_size),b''):h.update(chunk)
    return h.hexdigest()

def check_ssd_space(reserve_gib=12):
    free=shutil.disk_usage(LOCAL_ROOT).free
    already=sum(local_path_from_record(r).stat().st_size for r in records if local_path_from_record(r).is_file())
    # Conservador: espaço total necessário menos dados já presentes, mais reserva.
    need=max(0,TOTAL_BYTES-already)+int(reserve_gib*2**30)
    print(f'SSD free={free/2**30:.2f} GiB | remaining estimate={(TOTAL_BYTES-already)/2**30:.2f} GiB | reserve={reserve_gib} GiB',flush=True)
    assert free>=need, 'SSD insuficiente: libere espaço ou use runtime com mais armazenamento'

check_ssd_space()
print('SPACE PREFLIGHT: PASS')

In [ ]:
# Execute esta célula para iniciar/continuar staging. Não requer GPU.
# Em cada runtime novo, execute novamente. NÃO há acesso a shards pelo treino antes do PASS.
def stage_shards():
    t0=time.monotonic();last=t0;copied=0;reused=0;bytes_copied=0
    for i,r in enumerate(records,1):
        src=Path(r[3]);dst=local_path_from_record(r)
        size=int(r[4])*BYTES_PER_BLOCK; expected=r[5].lower()
        if dst.is_file() and dst.stat().st_size==size and sha256_file(dst)==expected:
            reused+=1
        else:
            assert src.is_file(), f'Source missing: {src}'
            dst.parent.mkdir(parents=True,exist_ok=True)
            tmp=dst.with_name(dst.name+'.part')
            try:
                with src.open('rb') as fi, tmp.open('wb') as fo:
                    shutil.copyfileobj(fi,fo,length=8*1024*1024)
                    fo.flush();os.fsync(fo.fileno())
                assert tmp.stat().st_size==size, f'Bad size: {src}'
                assert sha256_file(tmp)==expected, f'Bad SHA256: {src}'
                os.replace(tmp,dst)
            except Exception:
                tmp.unlink(missing_ok=True)
                raise
            copied+=1;bytes_copied+=size
        now=time.monotonic()
        if i%250==0 or now-last>=30 or i==len(records):
            print(f'{i:,}/{len(records):,} | copied={copied:,} reused={reused:,} | new_GiB={bytes_copied/2**30:.2f} | elapsed={now-t0:.1f}s',flush=True)
            last=now
    print('SSD STAGING COMPLETE:',copied,'copied,',reused,'reused',flush=True)
    return {'copied':copied,'reused':reused,'seconds':time.monotonic()-t0}

STAGING_RESULT=stage_shards()

## 3. Copiar SQLite para SSD e construir catálogo **local**

O banco original é preservado. A cópia local é reconstruída com os caminhos remapeados para `/content/maia/corpus_v8/shards`. O hash do catálogo original continua imutável; o hash do catálogo local será diferente por causa dos caminhos.

In [ ]:
def write_local_catalog():
    tmp=LOCAL_DB.with_suffix('.sqlite3.tmp')
    tmp.unlink(missing_ok=True)
    shutil.copy2(DRIVE_DB,tmp)
    con=sqlite3.connect(tmp)
    try:
        con.execute('BEGIN IMMEDIATE')
        for r in records:
            con.execute('UPDATE shard SET path=? WHERE source=? AND split=? AND shard_index=?',
                        (str(local_path_from_record(r)),r[0],r[1],r[2]))
        con.commit()
        local_count=con.execute('SELECT COUNT(*) FROM shard').fetchone()[0]
        assert local_count==len(records)
        paths=con.execute('SELECT path FROM shard').fetchall()
        assert all(Path(p[0]).is_relative_to(LOCAL_SHARDS) for p in paths)
    finally:con.close()
    os.replace(tmp,LOCAL_DB)
    return local_count

assert write_local_catalog()==60612
manifest={'schema_version':1,'catalog_sha256_original':CATALOG_HASH,
          'shards':len(records),'blocks':sum(int(r[4]) for r in records),
          'total_bytes':TOTAL_BYTES,'local_db':str(LOCAL_DB),
          'local_shards':str(LOCAL_SHARDS),'stage_complete':True,
          'source_db':str(DRIVE_DB)}
manifest_path=LOCAL_CORPUS/'ssd_stage_manifest.json'
manifest_path.write_text(json.dumps(manifest,indent=2))
print('LOCAL CATALOG READY:',LOCAL_DB)
print('SSD STAGING VERIFIED: PASS')

## 4. Leitor SSD-only e teste de retomada **sem GPU**

Rejeita qualquer caminho fora de `/content/maia/`. Os batches de inspeção são apenas matrizes de tokens; **não são enviados ao modelo**. A posição do sampler é um índice global determinístico. Esta etapa valida leitura, splits e retomada da sequência de blocos, sem consumir dados em treinamento.

In [ ]:
import bisect, random
class LocalShardReader:
    def __init__(self,db_path,split='train'):
        assert Path(db_path).resolve()==LOCAL_DB.resolve()
        self.con=sqlite3.connect(f'file:{LOCAL_DB}?mode=ro',uri=True)
        self.rows=self.con.execute('SELECT source,split,shard_index,path,blocks FROM shard WHERE split=? ORDER BY source,shard_index',(split,)).fetchall()
        assert self.rows
        self.starts=[0]
        for r in self.rows:
            assert Path(r[3]).resolve().is_relative_to(LOCAL_SHARDS.resolve()),'Remote path prohibited'
            self.starts.append(self.starts[-1]+int(r[4]))
        self.total=self.starts[-1]
    def block(self,ordinal):
        assert 0<=ordinal<self.total
        i=bisect.bisect_right(self.starts,ordinal)-1
        r=self.rows[i];block_idx=ordinal-self.starts[i]
        with open(r[3],'rb',buffering=0) as f:
            f.seek(block_idx*BYTES_PER_BLOCK)
            raw=f.read(BYTES_PER_BLOCK)
        assert len(raw)==BYTES_PER_BLOCK
        a=np.frombuffer(raw,dtype='<u4')
        assert int(a.max())<VOCAB_SIZE
        return (r[0],r[1],r[2],block_idx),a
    def close(self):self.con.close()

# Permutação afim por época: bijeção sobre todos os blocos, sem listas gigantes.
def permuted_ordinal(position,total,seed=20261105):
    epoch,slot=divmod(int(position),total)
    rng=random.Random(seed+epoch)
    a=rng.randrange(1,total)
    while math.gcd(a,total)!=1:a=(a+1)%total or 1
    b=rng.randrange(total)
    return (a*slot+b)%total

reader=LocalShardReader(LOCAL_DB)
assert reader.total==3390434

def batch_digest(start,count=32):
    h=hashlib.sha256();keys=[]
    for pos in range(start,start+count):
        key,tokens=reader.block(permuted_ordinal(pos,reader.total))
        keys.append(key)
        h.update(tokens.tobytes())
    return h.hexdigest(),keys

# Duas sequências: contínua (0..319) e interrompida em 160, depois retomada.
continuous=[batch_digest(pos)[0] for pos in range(0,320,32)]
interrupted=[batch_digest(pos)[0] for pos in range(0,160,32)]
restored_next_position=160
interrupted += [batch_digest(pos)[0] for pos in range(restored_next_position,320,32)]
assert continuous==interrupted
assert len(set(batch_digest(0)[1]))==32
print('SSD-ONLY REAL DATA READER: PASS')
print('REAL BATCH SEQUENCE RESUME: BITWISE MATCH')
print('Final next position:',320)
reader.close()

## 5. Checkpoints: SSD primeiro, Drive somente na sincronização

Estas funções são utilitários para o futuro motor de treinamento. Gravam checkpoint local de forma atômica e fazem cópia remota também atômica (arquivo temporário + renomeação). Em retomada, o checkpoint remoto é copiado primeiro ao SSD, com SHA-256 verificado. **Não executar `torch.load` em checkpoints não confiáveis.** O treino deve salvar `model`, `optimizer`, `scheduler`, `rng`, `next_position` e `catalog_sha256_original` juntos. O intervalo de sincronização deve ser escolhido conforme o risco de perder progresso entre checkpoints.

In [ ]:
def atomic_checkpoint_to_ssd(state,name):
    import torch
    assert name.endswith('.pt') and '/' not in name and '\\' not in name
    dst=LOCAL_CKPTS/name;tmp=LOCAL_CKPTS/(name+'.part')
    try:
        torch.save(state,tmp)
        with tmp.open('rb') as f:os.fsync(f.fileno())
        os.replace(tmp,dst)
    finally:tmp.unlink(missing_ok=True)
    return dst

def sync_checkpoint_to_drive(local_path):
    src=Path(local_path).resolve()
    assert src.is_file() and src.parent==LOCAL_CKPTS.resolve(),'Checkpoint must be on SSD'
    DRIVE_CKPTS.mkdir(parents=True,exist_ok=True)
    dst=DRIVE_CKPTS/src.name;tmp=DRIVE_CKPTS/(src.name+'.part')
    try:
        shutil.copyfile(src,tmp)
        assert sha256_file(src)==sha256_file(tmp),'Remote copy checksum mismatch'
        os.replace(tmp,dst)
    finally:tmp.unlink(missing_ok=True)
    print('PERSISTED:',dst,flush=True)
    return dst

def restore_checkpoint_to_ssd(name):
    assert name.endswith('.pt') and Path(name).name==name
    src=DRIVE_CKPTS/name
    assert src.is_file(),f'Checkpoint not found: {src}'
    dst=LOCAL_CKPTS/name;tmp=LOCAL_CKPTS/(name+'.part')
    try:
        shutil.copyfile(src,tmp)
        assert sha256_file(src)==sha256_file(tmp),'Checkpoint transfer corrupted'
        os.replace(tmp,dst)
    finally:tmp.unlink(missing_ok=True)
    return dst

def validate_checkpoint_identity(state):
    assert state['catalog_sha256_original']==CATALOG_HASH,'Corpus identity mismatch'
    assert state['next_position']==state['step']*state['blocks_per_update'],'Sampler position mismatch'
    return True

# Teste CPU de persistência com payload mínimo; não cria checkpoint de 5.67 GiB.
TEST_NAME='ssd_first_io_smoke_test.pt'
if (DRIVE_CKPTS/TEST_NAME).exists():
    print('Smoke test remoto já existe; verificando sem sobrescrever.')
    restored=restore_checkpoint_to_ssd(TEST_NAME)
    import torch
    previous=torch.load(restored,map_location='cpu',weights_only=True)
    assert previous['mode']=='io_test_only' and validate_checkpoint_identity(previous)
    print('EXISTING CHECKPOINT SMOKE TEST: PASS')
else:
    print('Criando novo checkpoint de teste (somente CPU).')
if not (DRIVE_CKPTS/TEST_NAME).exists():
    state={'step':5,'blocks_per_update':32,'next_position':160,
           'catalog_sha256_original':CATALOG_HASH,'mode':'io_test_only'}
    assert validate_checkpoint_identity(state)
    local=atomic_checkpoint_to_ssd(state,TEST_NAME)
    remote=sync_checkpoint_to_drive(local)
    local.unlink()
    restored=restore_checkpoint_to_ssd(TEST_NAME)
    import torch
    loaded=torch.load(restored,map_location='cpu',weights_only=True)
    assert loaded==state and validate_checkpoint_identity(loaded)
    print('SSD CHECKPOINT PERSISTENCE + RESTORE: PASS')

## 6. Gate final e próximos passos

Este notebook valida **cópia integral, integridade dos arquivos locais, caminhos SSD-only, leitura real e checkpoint de teste**. Não faz pré-treinamento. Para executar o Maia Lite 507M, o motor deve iniciar **após** este gate, apontar `DB` para `LOCAL_DB`, gravar checkpoints em `LOCAL_CKPTS`, e copiar checkpoints para Drive apenas nos momentos programados. **Não reutilizar** o checkpoint sintético como pesos iniciais do pré-treinamento definitivo.

O treinamento com shards reais continua **bloqueado até que exista uma lista de fontes e revisões aprovadas**. Não confundir sucesso do staging com aprovação de licença. Para sessões futuras: montar Drive → staging SSD → verificar → restaurar checkpoint ao SSD → inicializar GPU → continuar.

In [ ]:
assert LOCAL_DB.is_file() and manifest_path.is_file()
assert json.loads(manifest_path.read_text())['stage_complete'] is True
print('SSD-FIRST PREFLIGHT COMPLETE')
print('LOCAL_DB:',LOCAL_DB)
print('LOCAL_SHARDS:',LOCAL_SHARDS)
print('LOCAL_CKPTS:',LOCAL_CKPTS)
print('CATALOG_HASH:',CATALOG_HASH)
print('REAL TRAINING AUTHORIZED: NO (requires separate reviewed approval manifest)')